# CommissionLens: end-to-end pipeline

Commission-adjusted alpha prediction for Indian equity mutual funds. This notebook runs the full workflow: synthetic panel generation, feature engineering, the CommissionNet sequence model, SHAP explainability, and the SIP back-validation. It imports the same flat modules used by `main.py`, so it stays thin.

Run it from the repository root, top to bottom.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from config import load_config
from data import generate_synthetic_panel
from features import build_panel, TARGET_CLS, TARGET_REG
from model import build_sequences, temporal_split, subset_bundle, train_model, evaluate_model

config = load_config()
np.random.seed(config.seed)
config.data.n_funds, config.data.start_date, config.data.end_date

## 1. Data

Generate the synthetic panel of funds, NAV paths, fund attributes, and the macro block.

In [ ]:
source = generate_synthetic_panel(config.data.n_funds, config.data.start_date,
                                  config.data.end_date, config.data.risk_free_annual, config.seed)
print('funds:', source.nav_direct.shape[1], '| daily NAV rows:', source.nav_direct.shape[0])
source.macro.head()

In [ ]:
ax = source.nav_direct.iloc[:, :4].plot(figsize=(9, 4), title='Sample direct-plan NAV paths')
ax.set_ylabel('NAV')
plt.tight_layout(); plt.show()

## 2. Feature engineering

Build the quarterly panel: trailing risk metrics, fund attributes, macro regime, and the forward-looking net-alpha target.

In [ ]:
panel = build_panel(source, config)
print('fund-quarter rows:', len(panel))
print('commission-justified base rate:', round(panel[TARGET_CLS].mean(), 3))
panel.head()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(panel[TARGET_REG], bins=40, color='#c9a227')
ax.axvline(0, color='black', linewidth=1)
ax.set_title('Distribution of next-quarter net alpha'); ax.set_xlabel('net alpha (annualised)')
plt.tight_layout(); plt.show()

## 3. Sequences, temporal split, and training

Each sample is a window of consecutive quarters for one fund. CommissionNet is a bidirectional GRU with attention pooling and joint regression plus classification heads.

In [ ]:
bundle = build_sequences(panel, config.model.sequence_length)
train, val, test = temporal_split(bundle, config.model.test_fraction, config.model.val_fraction)
print('sample shape:', bundle.features.shape, '| train/val/test:', len(train), len(val), len(test))

model, history = train_model(train, val, config.model)
hist = pd.DataFrame(history)
ax = hist.plot(x='epoch', y='val_loss', figsize=(7, 4), title='Validation loss')
ax.set_ylabel('loss'); plt.tight_layout(); plt.show()

## 4. Evaluation

Regression (RMSE, R^2) and classification (AUC-ROC, F1, precision at the top decile) on held-out quarters.

In [ ]:
pd.Series(evaluate_model(model, test, config.simulation.top_decile))

## 5. SHAP explainability

Which features drive the commission-justification call?

In [ ]:
from explain import compute_feature_importance

importance = compute_feature_importance(model, train.features, test.features)
top = importance.head(8)[::-1]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top['feature'], top['mean_abs_shap'], color='#c9a227')
ax.set_xlabel('mean |SHAP|'); ax.set_title('Most predictive features of commission justification')
plt.tight_layout(); plt.show()
importance.head(5)

## 6. SIP back-validation

Compare a model-guided SIP against naive regular-plan investing. The model here is trained only on quarters that close before the simulation window.

In [ ]:
from simulation import build_selections, build_all_fund_selections, run_sip

quarters = bundle.meta['quarter_end'].to_numpy()
pool = quarters < np.datetime64(config.simulation.start_date)
pool_quarters = np.sort(np.unique(quarters[pool]))
n_val = max(1, int(round(len(pool_quarters) * config.model.val_fraction)))
val_mask = np.isin(quarters, pool_quarters[-n_val:])
sim_model, _ = train_model(subset_bundle(bundle, pool & ~val_mask), subset_bundle(bundle, val_mask), config.model)

_, prob = sim_model.predict(bundle.features)
predictions = bundle.meta.copy(); predictions['probability'] = prob
predictions = predictions[pd.to_datetime(predictions['quarter_end']) < config.simulation.end_date]

guided = run_sip(source.nav_regular, build_selections(predictions, config.simulation.top_decile),
                 config.simulation.monthly_investment, config.simulation.start_date, config.simulation.end_date)
naive = run_sip(source.nav_regular, build_all_fund_selections(predictions),
                config.simulation.monthly_investment, config.simulation.start_date, config.simulation.end_date)
pd.DataFrame({'model_guided': [guided.xirr, guided.final_value],
              'naive_regular': [naive.xirr, naive.final_value]}, index=['xirr', 'final_value'])

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.bar(['Model-guided', 'Naive regular'], [guided.xirr * 100, naive.xirr * 100], color=['#c9a227', '#6b7280'])
ax.set_ylabel('SIP XIRR (%)'); ax.set_title('Commission-aware selection vs naive regular investing')
plt.tight_layout(); plt.show()

## 7. Takeaways

- CommissionNet frames commission justification as a joint regression and classification problem over quarterly fund sequences.
- SHAP surfaces the information ratio, Sharpe, and rolling alpha (skill-persistence signals) as the primary drivers.
- The SIP back-validation quantifies the rupee value of commission-aware selection versus naive regular investing.

See `docs/report.md` for the written report.